In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")


Mounted at /content/drive
✅ Google Drive mounted!


In [ ]:
!pip install groq datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 9.7 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from groq import Groq
from datasets import load_dataset
import re
import time
import json
import os
import random
import threading

# ── Load all available API keys from Colab Secrets ───────────────
api_keys = []
for i in range(2, 7):
    try:
        key = userdata.get(f'GROQ_API_KEY_{i}')
        if key:
            api_keys.append(key)
            print(f" Loaded GROQ_API_KEY_{i}")
    except Exception:
        pass

if not api_keys:
    raise Exception(
        "Check Api Keys"
    )

print(f"\n Running with {len(api_keys)} API key(s) in parallel!")

# Create one Groq client per key
clients = [Groq(api_key=key) for key in api_keys]

# ── Models ────────────────────────────────────────────────────────
MODELS = {
    "LLaMA3.1-8B":  "llama-3.1-8b-instant",
    "LLaMA4-Scout": "meta-llama/llama-4-scout-17b-16e-instruct"
}

QUESTIONS_PER_TECHNIQUE = 500
PROGRESS_FILE = "/content/drive/MyDrive/experiment_progress_final.json"

# ── Load GSM8K ────────────────────────────────────────────────────
print("\nLoading GSM8K test dataset from HuggingFace...")
dataset = load_dataset("openai/gsm8k", "main", split="test")

def parse_gsm8k_answer(answer_str):
    match = re.search(r'####\s*([\-0-9,\.]+)', answer_str)
    if match:
        return match.group(1).replace(",", "").strip()
    return "NO_ANSWER"

all_questions = []
for i, item in enumerate(dataset):
    all_questions.append({
        "id":       i + 1,
        "question": item["question"],
        "answer":   parse_gsm8k_answer(item["answer"])
    })

random.seed(42)
questions = random.sample(all_questions, QUESTIONS_PER_TECHNIQUE)
for i, q in enumerate(questions):
    q["id"] = i + 1

TOTAL = len(questions)
print(f"Loaded {len(all_questions)} questions, sampled {TOTAL} for experiment.")
print(f"Split across {len(api_keys)} parallel workers\n")

# ── Prompting Functions ───────────────────────────────────────────

def zero_shot_prompt(question):
    return (
        f"Answer the following maths question. "
        f"Give only the final numerical answer.\n\n"
        f"Question: {question}\n\nAnswer:"
    )

def chain_of_thought_prompt(question):
    return (
        f"Answer the following maths question. Think step by step, "
        f"then give your final numerical answer at the end on a new line "
        f"starting with 'Final Answer:'.\n\nQuestion: {question}"
    )

def tree_of_thoughts_prompt(question):
    return f"""You are solving a maths problem using a structured reasoning approach.

Question: {question}

Step 1 - Generate THREE different approaches to solving this problem:
Approach A: [write approach A]
Approach B: [write approach B]
Approach C: [write approach C]

Step 2 - Evaluate each approach and select the most reliable one.

Step 3 - Solve using the best approach step by step.

Final Answer: [write only the number here]"""

# ── Call Model With Retry ─────────────────────────────────────────

def call_model(client, model_id, prompt, temperature=0, max_retries=10):
    attempt = 0
    while attempt < max_retries:
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=512
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            attempt += 1
            error_str = str(e)
            if "429" in error_str:
                reason = "Rate limit (429)"
            elif "503" in error_str or "502" in error_str:
                reason = "Server overload (5xx)"
            elif "timeout" in error_str.lower():
                reason = "Request timeout"
            elif "restricted" in error_str.lower():
                reason = "Account restricted — check console.groq.com"
            else:
                reason = f"Error: {error_str[:60]}"
            wait = min(2 ** attempt, 60)
            print(f"      ⚠  {reason} — retry {attempt}/{max_retries} in {wait}s...")
            time.sleep(wait)
    print(f"      ✖  Gave up after {max_retries} retries.")
    return "ERROR"

# ── Helpers ───────────────────────────────────────────────────────

def extract_number(text):
    if text == "ERROR":
        return "ERROR"
    final_match = re.search(r'[Ff]inal [Aa]nswer[:\s]+([\-0-9,\.]+)', text)
    if final_match:
        return final_match.group(1).replace(",", "").strip()
    numbers = re.findall(r'\b(\-?[0-9]+\.?[0-9]*)\b', text)
    if numbers:
        return numbers[-1]
    return "NO_ANSWER"

def check_answer(extracted, correct):
    if extracted in ("ERROR", "NO_ANSWER"):
        return False
    try:
        return round(float(extracted), 2) == round(float(correct), 2)
    except:
        return False

# ── Thread-Safe Progress Save/Load ───────────────────────────────
save_lock = threading.Lock()

def save_progress(results):
    with save_lock:
        with open(PROGRESS_FILE, "w") as f:
            json.dump(results, f, indent=2)

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        print(f" Found saved progress on Google Drive — resuming...")
        with open(PROGRESS_FILE, "r") as f:
            return json.load(f)
    print("No saved progress found — starting fresh.")
    return None

# ── Initialise Results ────────────────────────────────────────────

techniques = {
    "Zero-Shot":        zero_shot_prompt,
    "Chain-of-Thought": chain_of_thought_prompt,
    "Tree-of-Thoughts": tree_of_thoughts_prompt
}

saved = load_progress()
results = {}
results_lock = threading.Lock()

for model_name in MODELS:
    results[model_name] = {}
    for technique in techniques:
        if saved and model_name in saved and technique in saved[model_name]:
            results[model_name][technique] = saved[model_name][technique]
            done = len(results[model_name][technique]["details"])
            print(f"  Resuming {model_name} / {technique}: {done}/{TOTAL} already done")
        else:
            results[model_name][technique] = {
                "correct": 0,
                "total":   TOTAL,
                "details": []
            }

# ── Worker Function (runs in each thread) ────────────────────────

def worker(thread_id, client, model_name, model_id, technique_name,
           prompt_fn, question_chunk):
    tag = f"[Thread {thread_id+1} | {model_name} | {technique_name}]"

    for q in question_chunk:
        prompt     = prompt_fn(q["question"])
        response   = call_model(client, model_id, prompt)
        extracted  = extract_number(response)
        is_correct = check_answer(extracted, q["answer"])

        with results_lock:
            results[model_name][technique_name]["details"].append({
                "q_id":           q["id"],
                "question":       q["question"],
                "correct_answer": q["answer"],
                "model_answer":   extracted,
                "correct":        is_correct
            })
            if is_correct:
                results[model_name][technique_name]["correct"] += 1

            done_so_far   = len(results[model_name][technique_name]["details"])
            correct_count = results[model_name][technique_name]["correct"]

            save_progress(results)

        status = "✔" if is_correct else "✘"
        print(f"  {tag} Q{q['id']:03d} | "
              f"expected={q['answer']:<8} got={extracted:<10} [{status}] | "
              f"acc: {correct_count}/{done_so_far} "
              f"({(correct_count/done_so_far)*100:.1f}%)")

        time.sleep(4)

# ── Main Experiment Loop ──────────────────────────────────────────

n_keys = len(api_keys)

print("\n" + "=" * 70)
print("PROMPT ENGINEERING EXPERIMENT — PARALLEL EDITION")
print(f"Old Model : LLaMA 3.1 8B   (llama-3.1-8b-instant)              [2024]")
print(f"New Model : LLaMA 4 Scout  (llama-4-scout-17b-16e-instruct)    [2025]")
print(f"Techniques: Zero-Shot | Chain-of-Thought | Tree-of-Thoughts")
print(f"Dataset   : {TOTAL} questions per technique (seed=42)")
print(f"API Keys  : {n_keys} keys running in parallel")
print(f"Speed     : ~{round(3/n_keys, 1)}x faster than single key")
print(f"Progress  : Auto-saved to Google Drive after every question")
print("=" * 70)

for model_name, model_id in MODELS.items():
    print(f"\n>>> Running: {model_name}  ({model_id})")
    print("-" * 60)

    for technique_name, prompt_fn in techniques.items():

        already_done  = results[model_name][technique_name]["details"]
        done_ids      = {d["q_id"] for d in already_done}
        remaining     = [q for q in questions if q["id"] not in done_ids]

        if not remaining:
            correct = results[model_name][technique_name]["correct"]
            acc = (correct / TOTAL) * 100
            print(f"  ── {technique_name}: already complete — "
                  f"{correct}/{TOTAL} ({acc:.1f}%)")
            continue

        print(f"\n  ── Technique: {technique_name} "
              f"({len(already_done)} done, {len(remaining)} remaining) ──")
        print(f"  Splitting {len(remaining)} questions across {n_keys} threads...")

        chunks = [[] for _ in range(n_keys)]
        for i, q in enumerate(remaining):
            chunks[i % n_keys].append(q)

        for i, chunk in enumerate(chunks):
            print(f"    Thread {i+1}: {len(chunk)} questions "
                  f"(Key ending ...{api_keys[i][-6:]})")

        threads = []
        for i in range(n_keys):
            if not chunks[i]:
                continue
            t = threading.Thread(
                target=worker,
                args=(i, clients[i], model_name, model_id,
                      technique_name, prompt_fn, chunks[i]),
                daemon=True
            )
            threads.append(t)

        for t in threads:
            t.start()

        for t in threads:
            t.join()

        correct = results[model_name][technique_name]["correct"]
        acc = (correct / TOTAL) * 100
        print(f"\n  >> {technique_name} Final Accuracy: "
              f"{correct}/{TOTAL} ({acc:.1f}%)\n")

# ── Final Results Table ───────────────────────────────────────────

print("\n" + "=" * 70)
print("FINAL RESULTS TABLE")
print("=" * 70)
print(f"{'Technique':<22} {'LLaMA3.1-8B (OLD)':>22} {'LLaMA4-Scout (NEW)':>22}")
print("-" * 68)

for technique in techniques:
    s_old = results["LLaMA3.1-8B"][technique]["correct"]
    s_new = results["LLaMA4-Scout"][technique]["correct"]
    a_old = (s_old / TOTAL) * 100
    a_new = (s_new / TOTAL) * 100
    print(f"{technique:<22} {s_old}/{TOTAL} ({a_old:.1f}%)   "
          f"{s_new}/{TOTAL} ({a_new:.1f}%)")

print("=" * 70)

# ── Improvement Summary ───────────────────────────────────────────

print("\nIMPROVEMENT  (LLaMA4-Scout vs LLaMA3.1-8B)")
print("-" * 45)
for technique in techniques:
    s_old = results["LLaMA3.1-8B"][technique]["correct"]
    s_new = results["LLaMA4-Scout"][technique]["correct"]
    diff  = ((s_new - s_old) / TOTAL) * 100
    arrow = "▲" if diff > 0 else ("▼" if diff < 0 else "=")
    print(f"  {technique:<22} {arrow} {abs(diff):.1f}%")

print("=" * 70)
print(f"\n✅ Full results saved to Google Drive: {PROGRESS_FILE}")

✅ Loaded GROQ_API_KEY_2
✅ Loaded GROQ_API_KEY_3
✅ Loaded GROQ_API_KEY_4
✅ Loaded GROQ_API_KEY_5
✅ Loaded GROQ_API_KEY_6

🚀 Running with 5 API key(s) in parallel!

Loading GSM8K test dataset from HuggingFace...
✅ Loaded 1319 questions, sampled 500 for experiment.
🔀 Split across 5 parallel workers

✅ Found saved progress on Google Drive — resuming...
  Resuming LLaMA3.1-8B / Zero-Shot: 150/500 already done
  Resuming LLaMA3.1-8B / Chain-of-Thought: 150/500 already done
  Resuming LLaMA3.1-8B / Tree-of-Thoughts: 154/500 already done
  Resuming LLaMA4-Scout / Zero-Shot: 150/500 already done
  Resuming LLaMA4-Scout / Chain-of-Thought: 150/500 already done
  Resuming LLaMA4-Scout / Tree-of-Thoughts: 150/500 already done

PROMPT ENGINEERING EXPERIMENT — PARALLEL EDITION
Old Model : LLaMA 3.1 8B   (llama-3.1-8b-instant)              [2024]
New Model : LLaMA 4 Scout  (llama-4-scout-17b-16e-instruct)    [2025]
Techniques: Zero-Shot | Chain-of-Thought | Tree-of-Thoughts
Dataset   : 500 questions 